# PIPE 4 — Statistical Validation

Run this after PIPE 3. It performs robustness checks and statistical tests over exported metrics.

## Run order
- 0) Setup & configuration
- 1) Load required artifacts from previous pipe(s)
- 2) Run computations
- 3) Export tables/figures/logs to runs/ and artifacts/

## Notes
- This release contains **no embedded secrets** and **no stored outputs**.
- Configure paths and keys via environment variables or `.env`.


In [ ]:
# NOTE: This cell is part of the end-to-end pipeline. Read the markdown cells for context and required inputs.
import pandas as pd
import numpy as np
from pathlib import Path
from math import sqrt
from scipy.stats import norm

ROOT = Path("")
XLSX = ROOT / "paper_tables_final.xlsx"

df = pd.read_excel(XLSX, sheet_name="CEST_per_claim")
print("[OK] Loaded CEST_per_claim:", df.shape)
print("Columns:", list(df.columns))

# --- sanity
for c in ["file","mode","retriever","gap_id","doc_id","CEST"]:
    assert c in df.columns, f"Missing col: {c}"

df["CEST"] = pd.to_numeric(df["CEST"], errors="coerce").fillna(0.0)

# --- map configs (explicit for your observed modes)
def map_config(retriever, mode):
    r = str(retriever).upper().strip()
    m = str(mode).upper().strip()

    if r == "R1" and "STRUCT" in m and "GROUNDED" in m:
        return "A4"
    if r == "R0" and "GROUNDED_NO_STRUCT" in m:
        return "A2"
    if r == "R0" and "GROUNDED_STRUCT" in m:
        return "A2S"  # extra condition (not in paper), keep separate
    return f"{r}_{mode}"

df["config"] = df.apply(lambda x: map_config(x["retriever"], x["mode"]), axis=1)

print("\n=== CONFIG COUNTS (rows) ===")
print(df["config"].value_counts())

# --- define a "claim key"
# Since sheet doesn't have claim_id, we must build one WITHOUT inventing:
# use (file, gap_id, doc_id?) would be wrong because doc_id varies per claim-doc row.
# Better: assume each row corresponds to a claim-doc pair and claim identity = (file, gap_id, mode, retriever, claim_index_within_gap)
# BUT we don't have claim_index. So we must infer claim grouping:
# Most reliable grouping: (file, mode, retriever, gap_id) represents a single "gap", not a claim.
# Therefore this sheet likely already has one row per (gap, doc) not claim.
# Let's check duplicates:
print("\n=== DUPLICATE CHECK ===")
print("Unique (file,mode,retriever,gap_id):", df[["file","mode","retriever","gap_id"]].drop_duplicates().shape[0])
print("Total rows:", len(df))

# If unique gaps << rows, then it's gap-doc table, not claim-doc.
# We'll aggregate at GAP level instead (still supports stats, but for gap-level, not claim-level).
# We'll create a binary "has_support" per gap based on max CEST.
gkey = ["file","mode","retriever","gap_id","config"]
agg = df.groupby(gkey).agg(
    max_CEST=("CEST","max"),
    n_docs=("doc_id","nunique")
).reset_index()

print("\n=== GAP-LEVEL AGG (preview) ===")
print(agg.head())

# --- inspect CEST distribution to choose threshold
print("\n=== CEST DISTRIBUTION (max per gap) ===")
desc = agg["max_CEST"].describe(percentiles=[0.5,0.75,0.9,0.95,0.99])
print(desc)

# Choose threshold: start with >=1 if scale supports it; otherwise use median
if desc["max"] <= 1.0:
    thr = float(desc["50%"])  # median
    rule = "median"
else:
    thr = 1.0
    rule = ">=1.0"

print(f"\n[THRESHOLD] Using threshold={thr} (rule={rule}) for 'supported_gap' based on max_CEST")

agg["supported"] = (agg["max_CEST"] >= thr).astype(int)
agg["unsupported"] = 1 - agg["supported"]

# --- Wilson CI for proportions
def wilson_ci(k, n, alpha=0.05):
    if n == 0:
        return (np.nan, np.nan)
    z = norm.ppf(1 - alpha/2)
    p = k / n
    denom = 1 + z**2/n
    center = (p + z**2/(2*n)) / denom
    half = (z * sqrt((p*(1-p) + z**2/(4*n)) / n)) / denom
    return (center - half, center + half)

# --- aggregate per config (gap-level)
rows = []
for cfg, g in agg.groupby("config"):
    n = int(len(g))
    sup = int(g["supported"].sum())
    unsup = int(g["unsupported"].sum())
    icr = sup / n if n else np.nan
    ucr = unsup / n if n else np.nan
    icr_lo, icr_hi = wilson_ci(sup, n)
    ucr_lo, ucr_hi = wilson_ci(unsup, n)
    rows.append({
        "config": cfg,
        "n_units": n,
        "supported_units": sup,
        "unsupported_units": unsup,
        "ICR": icr,
        "ICR_95CI_low": icr_lo,
        "ICR_95CI_high": icr_hi,
        "UCR": ucr,
        "UCR_95CI_low": ucr_lo,
        "UCR_95CI_high": ucr_hi
    })

stats = pd.DataFrame(rows).sort_values("config")
print("\n=== GAP-LEVEL SUPPORT STATS (proxy for Phase III support) ===")
print(stats.to_string(index=False))

# Save outputs in root
out1 = ROOT / "phase3_gap_support_stats_from_excel.csv"
stats.to_csv(out1, index=False)
print("[WRITE]", out1)

# LaTeX rows
def fmt_ci(lo, hi):
    return f"[{lo:.2f}, {hi:.2f}]"

print("\n=== LaTeX rows (gap-level proxy) ===")
print("% Configuration & ICR (95% CI) & UCR (95% CI) & n_units \\\\")
for _, r in stats.iterrows():
    print(
        f"{r['config']} & {r['ICR']:.2f} ({fmt_ci(r['ICR_95CI_low'], r['ICR_95CI_high'])})"
        f" & {r['UCR']:.2f} ({fmt_ci(r['UCR_95CI_low'], r['UCR_95CI_high'])})"
        f" & {int(r['n_units'])} \\\\"
    )
